# 参加者別の習熟曲線：べき乗式と指数式の比較

practice の1〜400試行を対象に、C1（ベースライン）、C2（長距離）、C3（幅狭）の全参加者について、試行番号 N と core MT・1/TPe の関係を調べる。元の試行番号を維持し、同じ番号に有効な記録が複数あれば参加者内で平均する。10試行平均などの平滑化は行わない。

| 指標 | 終端値付きべき乗式 | 指数式 | 単位 |
|---|---|---|---|
| core MT | a + b N^(−c) | a + b exp(−cN) | ms |
| 1/TPe | Y∞ + (Y₁−Y∞) N^(−c) | Y∞ + (Y₁−Y∞) exp(−c(N−1)) | 秒 |

分析画面の各式の制約と初期値3組を使用する。MTは非増加曲線で、1/TPeでは初回値Y₁と漸近値Y∞の大小を固定しない。いずれの指標でも両モデルの自由パラメータは3個。各試行のcore TPeの逆数を求めてから集計する。基本除外は countedAsTrial=true かつ coreExitMs が有限。MT外れ値・休憩境界周辺の追加除外は行わず、successによる選別もしない。

同一参加者・同一指標の同じ観測点に対するRMSEを比較し、小さい方を記録する。R²も併記する。この順位は観測試行への当てはまりを示す。RMSEの丸め前の順位を保存し、数値計算上の微差は区別して表示する。

In [1]:
import csv
import math
import os
import sys
from collections import Counter, defaultdict
from html import escape
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'analysis.py').is_file() and (p/'inputCSV').is_dir())
out = root/'output/learning_report/individual_core_mt_inverse_tpe_20261126_182515'
for name in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS','VECLIB_MAXIMUM_THREADS'):
    os.environ[name] = '1'
sys.path.insert(0, str(root))
import numpy as np
from IPython.display import HTML, display
from analysis import load_rows, trial_observations, fit_regression_model

def save_csv(name, rows):
    with (out/name).open('w',encoding='utf-8-sig',newline='') as stream:
        writer = csv.DictWriter(stream,fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

def table(headers, rows):
    head = ''.join('<th>'+escape(str(x))+'</th>' for x in headers)
    body = ''.join('<tr>'+''.join('<td>'+escape(str(x))+'</td>' for x in row)+'</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

models = {'core MT':('asymptotic_power','exponential'), '1/TPe':('inverse_TPe_power','inverse_TPe_exponential')}
units = {'core MT':'ms','1/TPe':'s'}
conditions = ('C1','C2','C3')
groups = defaultdict(list)
expected = set()
for row in load_rows(root/'inputCSV'):
    if row['phase'] != 'practice' or row['condition'] not in conditions or not 1 <= row['n'] <= 400:
        continue
    key = row['condition'],row['participant']
    expected.add(key)
    if row['countedAsTrial'] and row['coreExitMs'] is not None:
        groups[key].append(row)
assert set(groups) == expected
assert len({pid for cond,pid in expected}) == 30
assert Counter(c for c,pid in groups) == {c:10 for c in conditions}

## 個人別の回帰

比較表は1人1行、係数・警告の詳細表は1人・1指標・1モデルにつき1行。差は「指数RMSE − べき乗RMSE」で、負なら指数式の誤差が小さい。元データの精度を踏まえた実質的な優劣のしきい値は設けない。表示上「同等*」は、RMSEの差が大きい方のRMSEの10⁻¹²以下という数値上の扱いを示す。統計的な同等性の判定は行わない。

In [2]:
def observations(active, metric):
    inverse = metric == '1/TPe'
    x,y = trial_observations(active,'coreMtMs','TPe' if inverse else 'MT',inverse=inverse)
    return np.asarray(x),np.asarray(y)

def check_fit(x, y, model, fit):
    p = fit['parameters']
    if model == 'asymptotic_power':
        predicted = p['a']+p['b']*x**(-p['c'])
    elif model == 'exponential':
        predicted = p['a']+p['b']*np.exp(-p['c']*x)
    else:
        weight = x**(-p['c']) if model=='inverse_TPe_power' else np.exp(-p['c']*(x-1))
        predicted = p['Yinf']+(p['Y1']-p['Yinf'])*weight
    errors = y-predicted
    rss = float(errors @ errors)
    tss = float(np.sum((y-y.mean())**2))
    assert fit['points'] == len(x) == len(set(x))
    np.testing.assert_allclose(fit['rmse'], np.sqrt(rss/len(y)), rtol=1e-11)
    if fit['r2'] is None:
        assert tss<=1e-12
    else:
        np.testing.assert_allclose(fit['r2'], 1-rss/tss, atol=1e-12)

details, comparisons = [], []
for (cond,pid), rows in sorted(groups.items()):
    person = {'条件':cond,'参加者ID':pid}
    for metric, pair in models.items():
        unit = units[metric]
        x,y = observations({pid:rows},metric)
        person[metric+'点数'] = len(x)
        errors = []
        for label,model in zip(('べき乗','指数'),pair,strict=True):
            fit = fit_regression_model(x,y,model)
            ok = fit['ok']
            if ok:
                check_fit(x,y,model,fit)
            params = fit.get('parameters',{})
            warning = fit['warning'] if ok else fit['message']
            rmse,r2 = fit.get('rmse'),fit.get('r2')
            details.append({'条件':cond,'参加者ID':pid,'指標':metric,'式':label,'モデル':model,'単位':unit,
                            '点数':len(x),'先頭試行':int(x[0]) if len(x) else None,'末尾試行':int(x[-1]) if len(x) else None,
                            **{k:params.get(k) for k in ('a','b','c','Y1','Yinf')},
                            'R²':r2,'RMSE':rmse,'収束':ok,'警告':warning})
            person[f'{metric} {label}RMSE ({unit})'] = rmse
            person[f'{metric} {label}R²'] = r2
            errors.append(rmse)
        power,exp = errors
        diff,rank,judgment = None,'判定不能','判定不能'
        if power is not None and exp is not None:
            diff = exp-power
            rank = '指数' if diff < 0 else ('べき乗' if diff > 0 else '同値')
            judgment = '同等*' if math.isclose(power,exp,rel_tol=1e-12,abs_tol=0.) else rank
        person[f'{metric} RMSE差 (指数−べき乗, {unit})'] = diff
        person[metric+' 丸め前の順位'] = rank
        person[metric+' 比較結果'] = judgment
    comparisons.append(person)

save_csv('participants.csv',comparisons)
save_csv('model_details.csv',details)
print(f"{len(comparisons)}名、{len(details)}回帰。収束 {sum(d['収束'] for d in details)}件。数値計算は1コア。")

30名、120回帰。収束 120件。数値計算は1コア。


## 条件別の人数

「同等*」と表示した参加者は、べき乗式・指数式それぞれの人数から除いて集計した。

In [3]:
counts = []
for metric in models:
    for cond in (*conditions,'全体'):
        selected = [p for p in comparisons if cond=='全体' or p['条件']==cond]
        number = Counter(p[metric+' 比較結果'] for p in selected)
        counts.append([metric,cond,len(selected),*(number[k] for k in ('べき乗','指数','同等*','判定不能'))])
table(['指標','条件','人数','べき乗','指数','同等*','判定不能'],counts)

指標,条件,人数,べき乗,指数,同等*,判定不能
core MT,C1,10,1,9,0,0
core MT,C2,10,1,9,0,0
core MT,C3,10,1,8,1,0
core MT,全体,30,3,26,1,0
1/TPe,C1,10,3,7,0,0
1/TPe,C2,10,2,8,0,0
1/TPe,C3,10,1,9,0,0
1/TPe,全体,30,6,24,0,0


## 参加者別の適合度

CSVには丸め前のRMSE・R²とRMSE差を保存している。係数、収束の成否、制約境界・推定不安定の警告は model_details.csv で確認できる。

In [4]:
def fmt(value, digits):
    return '—' if value is None else f'{value:.{digits}f}'

for metric in models:
    unit = units[metric]
    places = 3 if unit=='ms' else 7
    table(['条件','参加者ID',f'{metric} 点数',f'べき乗RMSE ({unit})',f'指数RMSE ({unit})',
           'べき乗R²','指数R²','比較結果'],
          [[p['条件'],p['参加者ID'],p[metric+'点数'],
            *(fmt(p[f'{metric} {m}RMSE ({unit})'],places) for m in ('べき乗','指数')),
            *(fmt(p[f'{metric} {m}R²'],4) for m in ('べき乗','指数')),p[metric+' 比較結果']]
           for p in comparisons])

条件,参加者ID,core MT 点数,べき乗RMSE (ms),指数RMSE (ms),べき乗R²,指数R²,比較結果
C1,011,398,231.736,226.884,0.2247,0.2568,指数
C1,012,400,191.920,199.151,0.3932,0.3466,べき乗
C1,015,399,135.901,128.575,0.2358,0.3160,指数
C1,022,400,67.598,59.208,0.6539,0.7345,指数
C1,024,399,144.881,132.940,0.6248,0.6841,指数
C1,025,400,60.155,55.985,0.5367,0.5987,指数
C1,027,400,168.218,167.640,0.0600,0.0665,指数
C1,028,400,198.626,194.033,0.1198,0.1600,指数
C1,029,400,100.232,94.837,0.5876,0.6308,指数
C1,030,399,125.768,120.058,0.2336,0.3016,指数


条件,参加者ID,1/TPe 点数,べき乗RMSE (s),指数RMSE (s),べき乗R²,指数R²,比較結果
C1,011,398,0.0202566,0.0200065,0.0969,0.1191,指数
C1,012,400,0.0215740,0.0215047,0.0447,0.0508,指数
C1,015,399,0.0133716,0.0131451,0.1129,0.1427,指数
C1,022,400,0.0084798,0.0087944,0.3137,0.2618,べき乗
C1,024,399,0.0169121,0.0165432,0.2803,0.3114,指数
C1,025,400,0.0070987,0.0068879,0.2006,0.2474,指数
C1,027,400,0.0155757,0.0155855,0.0387,0.0375,べき乗
C1,028,400,0.0141273,0.0141639,0.1810,0.1767,べき乗
C1,029,400,0.0119713,0.0115695,0.2479,0.2975,指数
C1,030,399,0.0140326,0.0139488,0.0994,0.1101,指数
